# Métricas DT

In [1]:

# =========================================================
# ÁRBOL DE DECISIÓN - BOOTSTRAPPING + MÉTRICAS DE SISTEMA
# =========================================================

import pandas as pd
import numpy as np
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import clone

# =========================================================
# MÉTRICAS DE SISTEMA
# =========================================================
from time import perf_counter
import os
import psutil

process = psutil.Process(os.getpid())
metricas_sistema = []

def memoria_gb():
    """
    Retorna la memoria RSS utilizada por el proceso actual, expresada en GB.
    """
    return process.memory_info().rss / 1024 / 1024 / 1024


def iniciar_medicion():
    """
    Inicia una medición de tiempo y memoria.
    """
    return perf_counter(), memoria_gb()


def cerrar_medicion(etapa, inicio_tiempo, inicio_memoria, modelo="dt", observacion=""):
    """
    Cierra una medición de tiempo y memoria para una etapa específica.
    Guarda la información en la lista metricas_sistema.
    """
    fin_tiempo = perf_counter()
    fin_memoria = memoria_gb()

    tiempo_segundos = fin_tiempo - inicio_tiempo

    fila = {
        "modelo": modelo,
        "etapa": etapa,
        "tiempo_segundos": round(tiempo_segundos, 2),
        "tiempo_minutos": round(tiempo_segundos / 60, 2),
        "memoria_inicio_gb": round(inicio_memoria, 4),
        "memoria_fin_gb": round(fin_memoria, 4),
        "diferencia_memoria_gb": round(fin_memoria - inicio_memoria, 4),
        "observacion": observacion
    }

    metricas_sistema.append(fila)

    print(
        f"[{etapa}] Tiempo: {fila['tiempo_minutos']} min | "
        f"Memoria inicio: {fila['memoria_inicio_gb']} GB | "
        f"Memoria fin: {fila['memoria_fin_gb']} GB"
    )


# =========================================================
# CONFIG
# =========================================================
RUTA_DATASET = "../1_data_processed/v3_dataset_post_chi.parquet"
RUTA_SPLIT = Path("../1_data_processed/split_indices")

N_SAMPLE = 100
N_ITER = 500
K_FOLDS = 5
BASE_SEED = 42
PROGRESS_EVERY = 25

MODELO_ID = "dt"
MODELO_NOMBRE = "Árbol de Decisión"

CARPETA_OUT = Path("../4_results/modelo_dt")
CARPETA_OUT.mkdir(parents=True, exist_ok=True)

CARPETA_METRICAS_SISTEMA = Path("../4_results/metricas_sistema")
CARPETA_METRICAS_SISTEMA.mkdir(parents=True, exist_ok=True)

MODELO = DecisionTreeClassifier(
    criterion="gini",
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=BASE_SEED,
    class_weight="balanced"
)

# Medición total del pipeline
inicio_total, memoria_total_inicio = iniciar_medicion()


# =========================================================
# HELPERS
# =========================================================
def evaluar_modelo(modelo, X_eval, y_eval):
    """
    Evalúa un modelo binario y retorna métricas de clasificación.
    """
    y_pred = modelo.predict(X_eval)
    tn, fp, fn, tp = confusion_matrix(y_eval, y_pred, labels=[0, 1]).ravel()

    acc = accuracy_score(y_eval, y_pred)
    prec = precision_score(y_eval, y_pred, zero_division=0)
    rec = recall_score(y_eval, y_pred, zero_division=0)
    f1 = f1_score(y_eval, y_pred, zero_division=0)

    roc = np.nan
    pr = np.nan

    try:
        if hasattr(modelo, "predict_proba"):
            y_proba = modelo.predict_proba(X_eval)[:, 1]
            roc = roc_auc_score(y_eval, y_proba)
            pr = average_precision_score(y_eval, y_proba)
    except Exception:
        pass

    return acc, prec, rec, f1, roc, pr, tp, fp, tn, fn


def obtener_valor_etapa(df_metricas, etapa, columna):
    """
    Obtiene el valor de una columna para una etapa específica.
    Si la etapa no existe, retorna NaN.
    """
    fila = df_metricas[df_metricas["etapa"] == etapa]

    if fila.empty:
        return np.nan

    return fila.iloc[0][columna]


# =========================================================
# CARGA Y PREPARACIÓN
# =========================================================
inicio_etapa, memoria_etapa = iniciar_medicion()

for ext_name in ["pandas.period", "pandas.interval"]:
    try:
        pa.unregister_extension_type(ext_name)
    except Exception:
        pass

cols = pq.read_schema(RUTA_DATASET).names
cols_signo = [c for c in cols if c.startswith("signo_zodiacal_")]

if not cols_signo:
    raise ValueError("No se encontraron columnas signo_zodiacal_*.")

df = pd.read_parquet(RUTA_DATASET, engine="pyarrow")
print("Dataset cargado:", df.shape)

# Compactar tipos
for c in df.columns:
    if c.startswith("signo_zodiacal_"):
        df[c] = df[c].astype("uint8")
    elif c == "ESTANCIA_DIAS":
        df[c] = df[c].astype("int32")
    elif pd.api.types.is_numeric_dtype(df[c]):
        df[c] = df[c].astype("uint8")

idx_train = np.load(RUTA_SPLIT / "idx_train.npy")

# Pasar a NumPy para acelerar el loop
X_all = df.drop(columns=cols_signo).to_numpy(copy=False)
df_train = df.iloc[idx_train].reset_index(drop=True)
X_train_all = X_all[idx_train]

skf = StratifiedKFold(
    n_splits=K_FOLDS,
    shuffle=True,
    random_state=BASE_SEED
)

cerrar_medicion(
    etapa="Carga y preparación de datos",
    inicio_tiempo=inicio_etapa,
    inicio_memoria=memoria_etapa,
    modelo=MODELO_ID,
    observacion=f"Dataset {df.shape}, N_SAMPLE={N_SAMPLE}, N_ITER={N_ITER}, K_FOLDS={K_FOLDS}"
)


# =========================================================
# ENTRENAMIENTO Y EVALUACIÓN
# =========================================================
inicio_etapa, memoria_etapa = iniciar_medicion()

resultados = []

for signo in cols_signo:
    inicio_signo, memoria_signo = iniciar_medicion()

    print(f"\n{MODELO_ID.upper()} -> {signo}")

    y_train_all = df_train[signo].to_numpy(dtype=np.uint8)

    pos_idx = np.where(y_train_all == 1)[0]
    neg_idx = np.where(y_train_all == 0)[0]

    if len(pos_idx) == 0 or len(neg_idx) == 0:
        print(f"Saltando {signo}")

        cerrar_medicion(
            etapa=f"Entrenamiento y evaluación - {signo}",
            inicio_tiempo=inicio_signo,
            inicio_memoria=memoria_signo,
            modelo=MODELO_ID,
            observacion="Signo saltado por ausencia de clase positiva o negativa"
        )

        continue

    n_pos = N_SAMPLE // 2
    n_neg = N_SAMPLE - n_pos

    resultados_signo = []

    for it in range(N_ITER):
        rng = np.random.default_rng(BASE_SEED + it + abs(hash(signo)) % 10000)

        # Bootstrap balanceado
        sample_pos = rng.choice(pos_idx, size=n_pos, replace=True)
        sample_neg = rng.choice(neg_idx, size=n_neg, replace=True)
        sample_idx = np.concatenate([sample_pos, sample_neg])
        rng.shuffle(sample_idx)

        X_boot = X_train_all[sample_idx]
        y_boot = y_train_all[sample_idx]

        fold_metrics = []

        # Validación cruzada k-fold
        for tr_idx, val_idx in skf.split(X_boot, y_boot):
            X_tr, X_val = X_boot[tr_idx], X_boot[val_idx]
            y_tr, y_val = y_boot[tr_idx], y_boot[val_idx]

            m_cv = clone(MODELO)
            m_cv.fit(X_tr, y_tr)

            fold_metrics.append(evaluar_modelo(m_cv, X_val, y_val))

        fold_metrics = np.array(fold_metrics, dtype=float)

        resultados_signo.append({
            "modelo": MODELO_ID,
            "signo": signo,
            "iter": it + 1,
            "pos_rate_bootstrap": float(y_boot.mean()),
            "cv_accuracy": np.nanmean(fold_metrics[:, 0]),
            "cv_precision": np.nanmean(fold_metrics[:, 1]),
            "cv_recall": np.nanmean(fold_metrics[:, 2]),
            "cv_f1": np.nanmean(fold_metrics[:, 3]),
            "cv_roc_auc": np.nanmean(fold_metrics[:, 4]),
            "cv_pr_auc": np.nanmean(fold_metrics[:, 5]),
            "cv_tp_mean": np.nanmean(fold_metrics[:, 6]),
            "cv_fp_mean": np.nanmean(fold_metrics[:, 7]),
            "cv_tn_mean": np.nanmean(fold_metrics[:, 8]),
            "cv_fn_mean": np.nanmean(fold_metrics[:, 9]),
        })

        if (it + 1) % PROGRESS_EVERY == 0:
            print(f"   Iter {it + 1}/{N_ITER}")

    # Guardado parcial por signo
    df_signo = pd.DataFrame(resultados_signo)
    df_signo.to_csv(CARPETA_OUT / f"{MODELO_ID}_detalle_{signo}.csv", index=False)

    resultados.extend(resultados_signo)

    cerrar_medicion(
        etapa=f"Entrenamiento y evaluación - {signo}",
        inicio_tiempo=inicio_signo,
        inicio_memoria=memoria_signo,
        modelo=MODELO_ID,
        observacion=f"{N_ITER} iteraciones, CV={K_FOLDS}, N_SAMPLE={N_SAMPLE}"
    )

cerrar_medicion(
    etapa="Entrenamiento y evaluación total",
    inicio_tiempo=inicio_etapa,
    inicio_memoria=memoria_etapa,
    modelo=MODELO_ID,
    observacion=f"{len(cols_signo)} signos, {N_ITER} iteraciones por signo, CV={K_FOLDS}, N_SAMPLE={N_SAMPLE}"
)


# =========================================================
# GUARDADO FINAL DE RESULTADOS DEL MODELO
# =========================================================
inicio_etapa, memoria_etapa = iniciar_medicion()

df_detalle = pd.DataFrame(resultados)

detalle_path = CARPETA_OUT / f"{MODELO_ID}_detalle.csv"
resumen_path = CARPETA_OUT / f"{MODELO_ID}_resumen.csv"

df_detalle.to_csv(detalle_path, index=False)

cols_metricas = [
    "pos_rate_bootstrap",
    "cv_accuracy",
    "cv_precision",
    "cv_recall",
    "cv_f1",
    "cv_roc_auc",
    "cv_pr_auc",
    "cv_tp_mean",
    "cv_fp_mean",
    "cv_tn_mean",
    "cv_fn_mean"
]

if not df_detalle.empty:
    df_resumen = (
        df_detalle
        .groupby("signo")[cols_metricas]
        .agg(["mean", "std"])
    )

    df_resumen.to_csv(resumen_path)

    print(f"{MODELO_ID.upper()} terminado")
    print("Detalle:", detalle_path)
    print("Resumen:", resumen_path)

else:
    print("No hubo resultados para guardar.")

cerrar_medicion(
    etapa="Guardado final de resultados",
    inicio_tiempo=inicio_etapa,
    inicio_memoria=memoria_etapa,
    modelo=MODELO_ID,
    observacion="Exportación de detalle y resumen"
)


# =========================================================
# GUARDADO MÉTRICAS DE SISTEMA
# =========================================================
cerrar_medicion(
    etapa="Pipeline completo",
    inicio_tiempo=inicio_total,
    inicio_memoria=memoria_total_inicio,
    modelo=MODELO_ID,
    observacion="Tiempo total desde configuración hasta guardado final"
)

df_metricas_sistema = pd.DataFrame(metricas_sistema)

# Guardado dentro de la carpeta del modelo
metricas_path_modelo = CARPETA_OUT / f"{MODELO_ID}_metricas_sistema.csv"
df_metricas_sistema.to_csv(
    metricas_path_modelo,
    index=False,
    encoding="utf-8-sig"
)

# Guardado dentro de carpeta común para comparación entre modelos
metricas_path_comun = CARPETA_METRICAS_SISTEMA / f"{MODELO_ID}_metricas_sistema.csv"
df_metricas_sistema.to_csv(
    metricas_path_comun,
    index=False,
    encoding="utf-8-sig"
)


# =========================================================
# RESUMEN DE MÉTRICAS DE SISTEMA PARA ANEXO
# =========================================================
n_signos_procesados = df_detalle["signo"].nunique() if not df_detalle.empty else 0
n_observaciones_resultado = len(df_detalle)
n_iteraciones_totales = n_signos_procesados * N_ITER

tiempo_entrenamiento_total_seg = obtener_valor_etapa(
    df_metricas_sistema,
    "Entrenamiento y evaluación total",
    "tiempo_segundos"
)

tiempo_pipeline_total_seg = obtener_valor_etapa(
    df_metricas_sistema,
    "Pipeline completo",
    "tiempo_segundos"
)

memoria_inicio_pipeline_gb = obtener_valor_etapa(
    df_metricas_sistema,
    "Pipeline completo",
    "memoria_inicio_gb"
)

memoria_fin_pipeline_gb = obtener_valor_etapa(
    df_metricas_sistema,
    "Pipeline completo",
    "memoria_fin_gb"
)

diferencia_memoria_pipeline_gb = obtener_valor_etapa(
    df_metricas_sistema,
    "Pipeline completo",
    "diferencia_memoria_gb"
)

tiempo_promedio_iteracion_seg = (
    tiempo_entrenamiento_total_seg / n_iteraciones_totales
    if n_iteraciones_totales > 0 else np.nan
)

df_resumen_sistema = pd.DataFrame([{
    "modelo": MODELO_ID,
    "modelo_nombre": MODELO_NOMBRE,
    "n_sample": N_SAMPLE,
    "n_iter_por_signo": N_ITER,
    "k_folds": K_FOLDS,
    "n_signos_procesados": n_signos_procesados,
    "n_iteraciones_totales": n_iteraciones_totales,
    "n_observaciones_resultado": n_observaciones_resultado,
    "tiempo_entrenamiento_total_seg": round(tiempo_entrenamiento_total_seg, 2),
    "tiempo_entrenamiento_total_min": round(tiempo_entrenamiento_total_seg / 60, 2),
    "tiempo_pipeline_total_seg": round(tiempo_pipeline_total_seg, 2),
    "tiempo_pipeline_total_min": round(tiempo_pipeline_total_seg / 60, 2),
    "tiempo_promedio_iteracion_seg": round(tiempo_promedio_iteracion_seg, 4),
    "memoria_inicio_pipeline_gb": memoria_inicio_pipeline_gb,
    "memoria_fin_pipeline_gb": memoria_fin_pipeline_gb,
    "diferencia_memoria_pipeline_gb": diferencia_memoria_pipeline_gb,
    "observacion": "Métricas calculadas a partir de tiempo de ejecución y memoria RSS observada por proceso."
}])

resumen_path_modelo = CARPETA_OUT / f"{MODELO_ID}_metricas_sistema_resumen.csv"
resumen_path_comun = CARPETA_METRICAS_SISTEMA / f"{MODELO_ID}_metricas_sistema_resumen.csv"

df_resumen_sistema.to_csv(
    resumen_path_modelo,
    index=False,
    encoding="utf-8-sig"
)

df_resumen_sistema.to_csv(
    resumen_path_comun,
    index=False,
    encoding="utf-8-sig"
)

print("\nMétricas de sistema guardadas en:")
print(metricas_path_modelo)
print(metricas_path_comun)

print("\nResumen de métricas de sistema guardado en:")
print(resumen_path_modelo)
print(resumen_path_comun)

display(df_metricas_sistema)
display(df_resumen_sistema)

Dataset cargado: (5808498, 488)
[Carga y preparación de datos] Tiempo: 5.47 min | Memoria inicio: 0.1882 GB | Memoria fin: 11.4079 GB

DT -> signo_zodiacal_acuario
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500
   Iter 375/500
   Iter 400/500
   Iter 425/500
   Iter 450/500
   Iter 475/500
   Iter 500/500
[Entrenamiento y evaluación - signo_zodiacal_acuario] Tiempo: 0.62 min | Memoria inicio: 11.4109 GB | Memoria fin: 11.1292 GB

DT -> signo_zodiacal_aries
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500
   Iter 375/500
   Iter 400/500
   Iter 425/500
   Iter 450/500
   Iter 475/500
   Iter 500/500
[Entrenamiento y evaluación - signo_zodiacal_ari

,modelo,etapa,tiempo_segundos,tiempo_minutos,memoria_inicio_gb,memoria_fin_gb,diferencia_memoria_gb,observacion
0,dt,Carga y preparación de datos,327.94,5.47,0.1882,11.4079,11.2197,"Dataset (5808498, 488), N_SAMPLE=100, N_ITER=5..."
1,dt,Entrenamiento y evaluación - signo_zodiacal_ac...,37.24,0.62,11.4109,11.1292,-0.2818,"500 iteraciones, CV=5, N_SAMPLE=100"
2,dt,Entrenamiento y evaluación - signo_zodiacal_aries,36.51,0.61,11.1292,10.6136,-0.5156,"500 iteraciones, CV=5, N_SAMPLE=100"
3,dt,Entrenamiento y evaluación - signo_zodiacal_ca...,36.11,0.60,10.6136,10.5025,-0.1111,"500 iteraciones, CV=5, N_SAMPLE=100"
4,dt,Entrenamiento y evaluación - signo_zodiacal_ca...,38.71,0.65,10.5025,9.1475,-1.3550,"500 iteraciones, CV=5, N_SAMPLE=100"
5,dt,Entrenamiento y evaluación - signo_zodiacal_es...,38.33,0.64,9.1475,8.7536,-0.3938,"500 iteraciones, CV=5, N_SAMPLE=100"
6,dt,Entrenamiento y evaluación - signo_zodiacal_ge...,36.75,0.61,8.7536,8.6673,-0.0864,"500 iteraciones, CV=5, N_SAMPLE=100"
7,dt,Entrenamiento y evaluación - signo_zodiacal_leo,35.11,0.59,8.6673,5.9584,-2.7088,"500 iteraciones, CV=5, N_SAMPLE=100"
8,dt,Entrenamiento y evaluación - signo_zodiacal_libra,28.30,0.47,5.9584,6.1780,0.2195,"500 iteraciones, CV=5, N_SAMPLE=100"
9,dt,Entrenamiento y evaluación - signo_zodiacal_pi...,29.28,0.49,6.1780,6.3908,0.2129,"500 iteraciones, CV=5, N_SAMPLE=100"


,modelo,modelo_nombre,n_sample,n_iter_por_signo,k_folds,n_signos_procesados,n_iteraciones_totales,n_observaciones_resultado,tiempo_entrenamiento_total_seg,tiempo_entrenamiento_total_min,tiempo_pipeline_total_seg,tiempo_pipeline_total_min,tiempo_promedio_iteracion_seg,memoria_inicio_pipeline_gb,memoria_fin_pipeline_gb,diferencia_memoria_pipeline_gb,observacion
0,dt,Árbol de Decisión,100,500,5,12,6000,6000,401.68,6.69,729.83,12.16,0.0669,0.1882,6.6242,6.436,Métricas calculadas a partir de tiempo de ejec...


# Métricas LR

In [1]:

# =========================================================
# REGRESIÓN LOGÍSTICA - BOOTSTRAPPING + MÉTRICAS DE SISTEMA
# =========================================================

import pandas as pd
import numpy as np
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression

# =========================================================
# MÉTRICAS DE SISTEMA
# =========================================================
from time import perf_counter
import os
import psutil

process = psutil.Process(os.getpid())
metricas_sistema = []


def memoria_gb():
    """
    Retorna la memoria RSS utilizada por el proceso actual, expresada en GB.
    """
    return process.memory_info().rss / 1024 / 1024 / 1024


def iniciar_medicion():
    """
    Inicia una medición de tiempo y memoria.
    """
    return perf_counter(), memoria_gb()


def cerrar_medicion(etapa, inicio_tiempo, inicio_memoria, modelo="lr", observacion=""):
    """
    Cierra una medición de tiempo y memoria para una etapa específica.
    Guarda la información en la lista metricas_sistema.
    """
    fin_tiempo = perf_counter()
    fin_memoria = memoria_gb()

    tiempo_segundos = fin_tiempo - inicio_tiempo

    fila = {
        "modelo": modelo,
        "etapa": etapa,
        "tiempo_segundos": round(tiempo_segundos, 2),
        "tiempo_minutos": round(tiempo_segundos / 60, 2),
        "memoria_inicio_gb": round(inicio_memoria, 4),
        "memoria_fin_gb": round(fin_memoria, 4),
        "diferencia_memoria_gb": round(fin_memoria - inicio_memoria, 4),
        "observacion": observacion
    }

    metricas_sistema.append(fila)

    print(
        f"[{etapa}] Tiempo: {fila['tiempo_minutos']} min | "
        f"Memoria inicio: {fila['memoria_inicio_gb']} GB | "
        f"Memoria fin: {fila['memoria_fin_gb']} GB"
    )


# =========================================================
# CONFIG
# =========================================================
RUTA_DATASET = "../1_data_processed/v3_dataset_post_chi.parquet"
RUTA_SPLIT = Path("../1_data_processed/split_indices")

N_SAMPLE = 100
N_ITER = 500
K_FOLDS = 5
BASE_SEED = 42
PROGRESS_EVERY = 25

MODELO_ID = "lr"
MODELO_NOMBRE = "Regresión Logística"

CARPETA_OUT = Path("../4_results/modelo_lr")
CARPETA_OUT.mkdir(parents=True, exist_ok=True)

CARPETA_METRICAS_SISTEMA = Path("../4_results/metricas_sistema")
CARPETA_METRICAS_SISTEMA.mkdir(parents=True, exist_ok=True)

LR_PARAMS = dict(
    solver="liblinear",
    C=1.0,
    max_iter=2000,
    random_state=BASE_SEED,
    class_weight="balanced"
)

MODELO = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(**LR_PARAMS))
])

# Medición total del pipeline
inicio_total, memoria_total_inicio = iniciar_medicion()


# =========================================================
# HELPERS
# =========================================================
def evaluar_modelo(modelo, X_eval, y_eval):
    """
    Evalúa un modelo binario y retorna métricas de clasificación.
    """
    y_pred = modelo.predict(X_eval)
    tn, fp, fn, tp = confusion_matrix(y_eval, y_pred, labels=[0, 1]).ravel()

    acc = accuracy_score(y_eval, y_pred)
    prec = precision_score(y_eval, y_pred, zero_division=0)
    rec = recall_score(y_eval, y_pred, zero_division=0)
    f1 = f1_score(y_eval, y_pred, zero_division=0)

    roc = np.nan
    pr = np.nan

    try:
        if hasattr(modelo, "predict_proba"):
            y_proba = modelo.predict_proba(X_eval)[:, 1]
            roc = roc_auc_score(y_eval, y_proba)
            pr = average_precision_score(y_eval, y_proba)
    except Exception:
        pass

    return acc, prec, rec, f1, roc, pr, tp, fp, tn, fn


def obtener_valor_etapa(df_metricas, etapa, columna):
    """
    Obtiene el valor de una columna para una etapa específica.
    Si la etapa no existe, retorna NaN.
    """
    fila = df_metricas[df_metricas["etapa"] == etapa]

    if fila.empty:
        return np.nan

    return fila.iloc[0][columna]


# =========================================================
# CARGA Y PREPARACIÓN
# =========================================================
inicio_etapa, memoria_etapa = iniciar_medicion()

for ext_name in ["pandas.period", "pandas.interval"]:
    try:
        pa.unregister_extension_type(ext_name)
    except Exception:
        pass

cols = pq.read_schema(RUTA_DATASET).names
cols_signo = [c for c in cols if c.startswith("signo_zodiacal_")]

if not cols_signo:
    raise ValueError("No se encontraron columnas signo_zodiacal_*.")

df = pd.read_parquet(RUTA_DATASET, engine="pyarrow")
print("Dataset cargado:", df.shape)

# Compactar tipos
for c in df.columns:
    if c.startswith("signo_zodiacal_"):
        df[c] = df[c].astype("uint8")
    elif c == "ESTANCIA_DIAS":
        df[c] = df[c].astype("int32")
    elif pd.api.types.is_numeric_dtype(df[c]):
        df[c] = df[c].astype("uint8")

idx_train = np.load(RUTA_SPLIT / "idx_train.npy")

# Pasar a NumPy para acelerar el loop
X_all = df.drop(columns=cols_signo).to_numpy(copy=False)
df_train = df.iloc[idx_train].reset_index(drop=True)
X_train_all = X_all[idx_train]

skf = StratifiedKFold(
    n_splits=K_FOLDS,
    shuffle=True,
    random_state=BASE_SEED
)

cerrar_medicion(
    etapa="Carga y preparación de datos",
    inicio_tiempo=inicio_etapa,
    inicio_memoria=memoria_etapa,
    modelo=MODELO_ID,
    observacion=f"Dataset {df.shape}, N_SAMPLE={N_SAMPLE}, N_ITER={N_ITER}, K_FOLDS={K_FOLDS}"
)


# =========================================================
# ENTRENAMIENTO Y EVALUACIÓN
# =========================================================
inicio_etapa, memoria_etapa = iniciar_medicion()

resultados = []

for signo in cols_signo:
    inicio_signo, memoria_signo = iniciar_medicion()

    print(f"\n{MODELO_ID.upper()} -> {signo}")

    y_train_all = df_train[signo].to_numpy(dtype=np.uint8)

    pos_idx = np.where(y_train_all == 1)[0]
    neg_idx = np.where(y_train_all == 0)[0]

    if len(pos_idx) == 0 or len(neg_idx) == 0:
        print(f"Saltando {signo}")

        cerrar_medicion(
            etapa=f"Entrenamiento y evaluación - {signo}",
            inicio_tiempo=inicio_signo,
            inicio_memoria=memoria_signo,
            modelo=MODELO_ID,
            observacion="Signo saltado por ausencia de clase positiva o negativa"
        )

        continue

    n_pos = N_SAMPLE // 2
    n_neg = N_SAMPLE - n_pos

    resultados_signo = []

    for it in range(N_ITER):
        rng = np.random.default_rng(BASE_SEED + it + abs(hash(signo)) % 10000)

        # Bootstrap balanceado
        sample_pos = rng.choice(pos_idx, size=n_pos, replace=True)
        sample_neg = rng.choice(neg_idx, size=n_neg, replace=True)
        sample_idx = np.concatenate([sample_pos, sample_neg])
        rng.shuffle(sample_idx)

        X_boot = X_train_all[sample_idx]
        y_boot = y_train_all[sample_idx]

        fold_metrics = []

        # Validación cruzada k-fold
        for tr_idx, val_idx in skf.split(X_boot, y_boot):
            X_tr, X_val = X_boot[tr_idx], X_boot[val_idx]
            y_tr, y_val = y_boot[tr_idx], y_boot[val_idx]

            m_cv = clone(MODELO)
            m_cv.fit(X_tr, y_tr)

            fold_metrics.append(evaluar_modelo(m_cv, X_val, y_val))

        fold_metrics = np.array(fold_metrics, dtype=float)

        resultados_signo.append({
            "modelo": MODELO_ID,
            "signo": signo,
            "iter": it + 1,
            "pos_rate_bootstrap": float(y_boot.mean()),
            "cv_accuracy": np.nanmean(fold_metrics[:, 0]),
            "cv_precision": np.nanmean(fold_metrics[:, 1]),
            "cv_recall": np.nanmean(fold_metrics[:, 2]),
            "cv_f1": np.nanmean(fold_metrics[:, 3]),
            "cv_roc_auc": np.nanmean(fold_metrics[:, 4]),
            "cv_pr_auc": np.nanmean(fold_metrics[:, 5]),
            "cv_tp_mean": np.nanmean(fold_metrics[:, 6]),
            "cv_fp_mean": np.nanmean(fold_metrics[:, 7]),
            "cv_tn_mean": np.nanmean(fold_metrics[:, 8]),
            "cv_fn_mean": np.nanmean(fold_metrics[:, 9]),
        })

        if (it + 1) % PROGRESS_EVERY == 0:
            print(f"   Iter {it + 1}/{N_ITER}")

    # Guardado parcial por signo
    df_signo = pd.DataFrame(resultados_signo)
    df_signo.to_csv(CARPETA_OUT / f"{MODELO_ID}_detalle_{signo}.csv", index=False)

    resultados.extend(resultados_signo)

    cerrar_medicion(
        etapa=f"Entrenamiento y evaluación - {signo}",
        inicio_tiempo=inicio_signo,
        inicio_memoria=memoria_signo,
        modelo=MODELO_ID,
        observacion=f"{N_ITER} iteraciones, CV={K_FOLDS}, N_SAMPLE={N_SAMPLE}"
    )

cerrar_medicion(
    etapa="Entrenamiento y evaluación total",
    inicio_tiempo=inicio_etapa,
    inicio_memoria=memoria_etapa,
    modelo=MODELO_ID,
    observacion=f"{len(cols_signo)} signos, {N_ITER} iteraciones por signo, CV={K_FOLDS}, N_SAMPLE={N_SAMPLE}"
)


# =========================================================
# GUARDADO FINAL DE RESULTADOS DEL MODELO
# =========================================================
inicio_etapa, memoria_etapa = iniciar_medicion()

df_detalle = pd.DataFrame(resultados)

detalle_path = CARPETA_OUT / f"{MODELO_ID}_detalle.csv"
resumen_path = CARPETA_OUT / f"{MODELO_ID}_resumen.csv"

df_detalle.to_csv(detalle_path, index=False)

cols_metricas = [
    "pos_rate_bootstrap",
    "cv_accuracy",
    "cv_precision",
    "cv_recall",
    "cv_f1",
    "cv_roc_auc",
    "cv_pr_auc",
    "cv_tp_mean",
    "cv_fp_mean",
    "cv_tn_mean",
    "cv_fn_mean"
]

if not df_detalle.empty:
    df_resumen = (
        df_detalle
        .groupby("signo")[cols_metricas]
        .agg(["mean", "std"])
    )

    df_resumen.to_csv(resumen_path)

    print(f"{MODELO_ID.upper()} terminado")
    print("Detalle:", detalle_path)
    print("Resumen:", resumen_path)

else:
    print("No hubo resultados para guardar.")

cerrar_medicion(
    etapa="Guardado final de resultados",
    inicio_tiempo=inicio_etapa,
    inicio_memoria=memoria_etapa,
    modelo=MODELO_ID,
    observacion="Exportación de detalle y resumen"
)


# =========================================================
# GUARDADO MÉTRICAS DE SISTEMA
# =========================================================
cerrar_medicion(
    etapa="Pipeline completo",
    inicio_tiempo=inicio_total,
    inicio_memoria=memoria_total_inicio,
    modelo=MODELO_ID,
    observacion="Tiempo total desde configuración hasta guardado final"
)

df_metricas_sistema = pd.DataFrame(metricas_sistema)

# Guardado dentro de la carpeta del modelo
metricas_path_modelo = CARPETA_OUT / f"{MODELO_ID}_metricas_sistema.csv"
df_metricas_sistema.to_csv(
    metricas_path_modelo,
    index=False,
    encoding="utf-8-sig"
)

# Guardado dentro de carpeta común para comparación entre modelos
metricas_path_comun = CARPETA_METRICAS_SISTEMA / f"{MODELO_ID}_metricas_sistema.csv"
df_metricas_sistema.to_csv(
    metricas_path_comun,
    index=False,
    encoding="utf-8-sig"
)


# =========================================================
# RESUMEN DE MÉTRICAS DE SISTEMA PARA ANEXO
# =========================================================
n_signos_procesados = df_detalle["signo"].nunique() if not df_detalle.empty else 0
n_observaciones_resultado = len(df_detalle)
n_iteraciones_totales = n_signos_procesados * N_ITER

tiempo_entrenamiento_total_seg = obtener_valor_etapa(
    df_metricas_sistema,
    "Entrenamiento y evaluación total",
    "tiempo_segundos"
)

tiempo_pipeline_total_seg = obtener_valor_etapa(
    df_metricas_sistema,
    "Pipeline completo",
    "tiempo_segundos"
)

memoria_inicio_pipeline_gb = obtener_valor_etapa(
    df_metricas_sistema,
    "Pipeline completo",
    "memoria_inicio_gb"
)

memoria_fin_pipeline_gb = obtener_valor_etapa(
    df_metricas_sistema,
    "Pipeline completo",
    "memoria_fin_gb"
)

diferencia_memoria_pipeline_gb = obtener_valor_etapa(
    df_metricas_sistema,
    "Pipeline completo",
    "diferencia_memoria_gb"
)

tiempo_promedio_iteracion_seg = (
    tiempo_entrenamiento_total_seg / n_iteraciones_totales
    if n_iteraciones_totales > 0 else np.nan
)

df_resumen_sistema = pd.DataFrame([{
    "modelo": MODELO_ID,
    "modelo_nombre": MODELO_NOMBRE,
    "n_sample": N_SAMPLE,
    "n_iter_por_signo": N_ITER,
    "k_folds": K_FOLDS,
    "n_signos_procesados": n_signos_procesados,
    "n_iteraciones_totales": n_iteraciones_totales,
    "n_observaciones_resultado": n_observaciones_resultado,
    "tiempo_entrenamiento_total_seg": round(tiempo_entrenamiento_total_seg, 2),
    "tiempo_entrenamiento_total_min": round(tiempo_entrenamiento_total_seg / 60, 2),
    "tiempo_pipeline_total_seg": round(tiempo_pipeline_total_seg, 2),
    "tiempo_pipeline_total_min": round(tiempo_pipeline_total_seg / 60, 2),
    "tiempo_promedio_iteracion_seg": round(tiempo_promedio_iteracion_seg, 4),
    "memoria_inicio_pipeline_gb": memoria_inicio_pipeline_gb,
    "memoria_fin_pipeline_gb": memoria_fin_pipeline_gb,
    "diferencia_memoria_pipeline_gb": diferencia_memoria_pipeline_gb,
    "observacion": "Métricas calculadas a partir de tiempo de ejecución y memoria RSS observada por proceso."
}])

resumen_path_modelo = CARPETA_OUT / f"{MODELO_ID}_metricas_sistema_resumen.csv"
resumen_path_comun = CARPETA_METRICAS_SISTEMA / f"{MODELO_ID}_metricas_sistema_resumen.csv"

df_resumen_sistema.to_csv(
    resumen_path_modelo,
    index=False,
    encoding="utf-8-sig"
)

df_resumen_sistema.to_csv(
    resumen_path_comun,
    index=False,
    encoding="utf-8-sig"
)

print("\nMétricas de sistema guardadas en:")
print(metricas_path_modelo)
print(metricas_path_comun)

print("\nResumen de métricas de sistema guardado en:")
print(resumen_path_modelo)
print(resumen_path_comun)

display(df_metricas_sistema)
display(df_resumen_sistema)


Dataset cargado: (5808498, 488)
[Carga y preparación de datos] Tiempo: 3.93 min | Memoria inicio: 0.1815 GB | Memoria fin: 12.2712 GB

LR -> signo_zodiacal_acuario
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500
   Iter 375/500
   Iter 400/500
   Iter 425/500
   Iter 450/500
   Iter 475/500
   Iter 500/500
[Entrenamiento y evaluación - signo_zodiacal_acuario] Tiempo: 0.67 min | Memoria inicio: 12.2747 GB | Memoria fin: 11.5811 GB

LR -> signo_zodiacal_aries
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500
   Iter 375/500
   Iter 400/500
   Iter 425/500
   Iter 450/500
   Iter 475/500
   Iter 500/500
[Entrenamiento y evaluación - signo_zodiacal_ari

,modelo,etapa,tiempo_segundos,tiempo_minutos,memoria_inicio_gb,memoria_fin_gb,diferencia_memoria_gb,observacion
0,lr,Carga y preparación de datos,235.58,3.93,0.1815,12.2712,12.0897,"Dataset (5808498, 488), N_SAMPLE=100, N_ITER=5..."
1,lr,Entrenamiento y evaluación - signo_zodiacal_ac...,40.23,0.67,12.2747,11.5811,-0.6936,"500 iteraciones, CV=5, N_SAMPLE=100"
2,lr,Entrenamiento y evaluación - signo_zodiacal_aries,38.47,0.64,11.5811,11.4501,-0.1310,"500 iteraciones, CV=5, N_SAMPLE=100"
3,lr,Entrenamiento y evaluación - signo_zodiacal_ca...,37.09,0.62,11.4501,9.8714,-1.5786,"500 iteraciones, CV=5, N_SAMPLE=100"
4,lr,Entrenamiento y evaluación - signo_zodiacal_ca...,35.03,0.58,9.8714,9.9899,0.1184,"500 iteraciones, CV=5, N_SAMPLE=100"
5,lr,Entrenamiento y evaluación - signo_zodiacal_es...,35.23,0.59,9.9899,9.8536,-0.1363,"500 iteraciones, CV=5, N_SAMPLE=100"
6,lr,Entrenamiento y evaluación - signo_zodiacal_ge...,34.66,0.58,9.8536,9.9606,0.1070,"500 iteraciones, CV=5, N_SAMPLE=100"
7,lr,Entrenamiento y evaluación - signo_zodiacal_leo,35.83,0.60,9.9606,9.9352,-0.0254,"500 iteraciones, CV=5, N_SAMPLE=100"
8,lr,Entrenamiento y evaluación - signo_zodiacal_libra,35.23,0.59,9.9352,4.5751,-5.3601,"500 iteraciones, CV=5, N_SAMPLE=100"
9,lr,Entrenamiento y evaluación - signo_zodiacal_pi...,33.20,0.55,4.5751,4.7895,0.2144,"500 iteraciones, CV=5, N_SAMPLE=100"


,modelo,modelo_nombre,n_sample,n_iter_por_signo,k_folds,n_signos_procesados,n_iteraciones_totales,n_observaciones_resultado,tiempo_entrenamiento_total_seg,tiempo_entrenamiento_total_min,tiempo_pipeline_total_seg,tiempo_pipeline_total_min,tiempo_promedio_iteracion_seg,memoria_inicio_pipeline_gb,memoria_fin_pipeline_gb,diferencia_memoria_pipeline_gb,observacion
0,lr,Regresión Logística,100,500,5,12,6000,6000,423.92,7.07,659.69,10.99,0.0707,0.1815,5.4027,5.2212,Métricas calculadas a partir de tiempo de ejec...


# Métricas RF

In [1]:

# =========================================================
# RANDOM FOREST - BOOTSTRAPPING + MÉTRICAS DE SISTEMA
# =========================================================

import pandas as pd
import numpy as np
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone

# =========================================================
# MÉTRICAS DE SISTEMA
# =========================================================
from time import perf_counter
import os
import psutil

process = psutil.Process(os.getpid())
metricas_sistema = []


def memoria_gb():
    """
    Retorna la memoria RSS utilizada por el proceso actual, expresada en GB.
    """
    return process.memory_info().rss / 1024 / 1024 / 1024


def iniciar_medicion():
    """
    Inicia una medición de tiempo y memoria.
    """
    return perf_counter(), memoria_gb()


def cerrar_medicion(etapa, inicio_tiempo, inicio_memoria, modelo="rf", observacion=""):
    """
    Cierra una medición de tiempo y memoria para una etapa específica.
    Guarda la información en la lista metricas_sistema.
    """
    fin_tiempo = perf_counter()
    fin_memoria = memoria_gb()

    tiempo_segundos = fin_tiempo - inicio_tiempo

    fila = {
        "modelo": modelo,
        "etapa": etapa,
        "tiempo_segundos": round(tiempo_segundos, 2),
        "tiempo_minutos": round(tiempo_segundos / 60, 2),
        "memoria_inicio_gb": round(inicio_memoria, 4),
        "memoria_fin_gb": round(fin_memoria, 4),
        "diferencia_memoria_gb": round(fin_memoria - inicio_memoria, 4),
        "observacion": observacion
    }

    metricas_sistema.append(fila)

    print(
        f"[{etapa}] Tiempo: {fila['tiempo_minutos']} min | "
        f"Memoria inicio: {fila['memoria_inicio_gb']} GB | "
        f"Memoria fin: {fila['memoria_fin_gb']} GB"
    )


# =========================================================
# CONFIG
# =========================================================
RUTA_DATASET = "../1_data_processed/v3_dataset_post_chi.parquet"
RUTA_SPLIT = Path("../1_data_processed/split_indices")

N_SAMPLE = 100
N_ITER = 500
K_FOLDS = 5
BASE_SEED = 42
PROGRESS_EVERY = 25

MODELO_ID = "rf"
MODELO_NOMBRE = "Random Forest"

CARPETA_OUT = Path("../4_results/modelo_rf")
CARPETA_OUT.mkdir(parents=True, exist_ok=True)

CARPETA_METRICAS_SISTEMA = Path("../4_results/metricas_sistema")
CARPETA_METRICAS_SISTEMA.mkdir(parents=True, exist_ok=True)

MODELO = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    n_jobs=-1,
    random_state=BASE_SEED,
    class_weight="balanced"
)

# Medición total del pipeline
inicio_total, memoria_total_inicio = iniciar_medicion()


# =========================================================
# HELPERS
# =========================================================
def evaluar_modelo(modelo, X_eval, y_eval):
    """
    Evalúa un modelo binario y retorna métricas de clasificación.
    """
    y_pred = modelo.predict(X_eval)
    tn, fp, fn, tp = confusion_matrix(y_eval, y_pred, labels=[0, 1]).ravel()

    acc = accuracy_score(y_eval, y_pred)
    prec = precision_score(y_eval, y_pred, zero_division=0)
    rec = recall_score(y_eval, y_pred, zero_division=0)
    f1 = f1_score(y_eval, y_pred, zero_division=0)

    roc = np.nan
    pr = np.nan

    try:
        if hasattr(modelo, "predict_proba"):
            y_proba = modelo.predict_proba(X_eval)[:, 1]
            roc = roc_auc_score(y_eval, y_proba)
            pr = average_precision_score(y_eval, y_proba)
    except Exception:
        pass

    return acc, prec, rec, f1, roc, pr, tp, fp, tn, fn


def obtener_valor_etapa(df_metricas, etapa, columna):
    """
    Obtiene el valor de una columna para una etapa específica.
    Si la etapa no existe, retorna NaN.
    """
    fila = df_metricas[df_metricas["etapa"] == etapa]

    if fila.empty:
        return np.nan

    return fila.iloc[0][columna]


# =========================================================
# CARGA Y PREPARACIÓN
# =========================================================
inicio_etapa, memoria_etapa = iniciar_medicion()

for ext_name in ["pandas.period", "pandas.interval"]:
    try:
        pa.unregister_extension_type(ext_name)
    except Exception:
        pass

cols = pq.read_schema(RUTA_DATASET).names
cols_signo = [c for c in cols if c.startswith("signo_zodiacal_")]

if not cols_signo:
    raise ValueError("No se encontraron columnas signo_zodiacal_*.")

df = pd.read_parquet(RUTA_DATASET, engine="pyarrow")
print("Dataset cargado:", df.shape)

# Compactar tipos
for c in df.columns:
    if c.startswith("signo_zodiacal_"):
        df[c] = df[c].astype("uint8")
    elif c == "ESTANCIA_DIAS":
        df[c] = df[c].astype("int32")
    elif pd.api.types.is_numeric_dtype(df[c]):
        df[c] = df[c].astype("uint8")

idx_train = np.load(RUTA_SPLIT / "idx_train.npy")

# Pasar a NumPy para acelerar el loop
X_all = df.drop(columns=cols_signo).to_numpy(copy=False)
df_train = df.iloc[idx_train].reset_index(drop=True)
X_train_all = X_all[idx_train]

skf = StratifiedKFold(
    n_splits=K_FOLDS,
    shuffle=True,
    random_state=BASE_SEED
)

cerrar_medicion(
    etapa="Carga y preparación de datos",
    inicio_tiempo=inicio_etapa,
    inicio_memoria=memoria_etapa,
    modelo=MODELO_ID,
    observacion=f"Dataset {df.shape}, N_SAMPLE={N_SAMPLE}, N_ITER={N_ITER}, K_FOLDS={K_FOLDS}"
)


# =========================================================
# ENTRENAMIENTO Y EVALUACIÓN
# =========================================================
inicio_etapa, memoria_etapa = iniciar_medicion()

resultados = []

for signo in cols_signo:
    inicio_signo, memoria_signo = iniciar_medicion()

    print(f"\n{MODELO_ID.upper()} -> {signo}")

    y_train_all = df_train[signo].to_numpy(dtype=np.uint8)

    pos_idx = np.where(y_train_all == 1)[0]
    neg_idx = np.where(y_train_all == 0)[0]

    if len(pos_idx) == 0 or len(neg_idx) == 0:
        print(f"Saltando {signo}")

        cerrar_medicion(
            etapa=f"Entrenamiento y evaluación - {signo}",
            inicio_tiempo=inicio_signo,
            inicio_memoria=memoria_signo,
            modelo=MODELO_ID,
            observacion="Signo saltado por ausencia de clase positiva o negativa"
        )

        continue

    n_pos = N_SAMPLE // 2
    n_neg = N_SAMPLE - n_pos

    resultados_signo = []

    for it in range(N_ITER):
        rng = np.random.default_rng(BASE_SEED + it + abs(hash(signo)) % 10000)

        # Bootstrap balanceado
        sample_pos = rng.choice(pos_idx, size=n_pos, replace=True)
        sample_neg = rng.choice(neg_idx, size=n_neg, replace=True)
        sample_idx = np.concatenate([sample_pos, sample_neg])
        rng.shuffle(sample_idx)

        X_boot = X_train_all[sample_idx]
        y_boot = y_train_all[sample_idx]

        fold_metrics = []

        # Validación cruzada k-fold
        for tr_idx, val_idx in skf.split(X_boot, y_boot):
            X_tr, X_val = X_boot[tr_idx], X_boot[val_idx]
            y_tr, y_val = y_boot[tr_idx], y_boot[val_idx]

            m_cv = clone(MODELO)
            m_cv.fit(X_tr, y_tr)

            fold_metrics.append(evaluar_modelo(m_cv, X_val, y_val))

        fold_metrics = np.array(fold_metrics, dtype=float)

        resultados_signo.append({
            "modelo": MODELO_ID,
            "signo": signo,
            "iter": it + 1,
            "pos_rate_bootstrap": float(y_boot.mean()),
            "cv_accuracy": np.nanmean(fold_metrics[:, 0]),
            "cv_precision": np.nanmean(fold_metrics[:, 1]),
            "cv_recall": np.nanmean(fold_metrics[:, 2]),
            "cv_f1": np.nanmean(fold_metrics[:, 3]),
            "cv_roc_auc": np.nanmean(fold_metrics[:, 4]),
            "cv_pr_auc": np.nanmean(fold_metrics[:, 5]),
            "cv_tp_mean": np.nanmean(fold_metrics[:, 6]),
            "cv_fp_mean": np.nanmean(fold_metrics[:, 7]),
            "cv_tn_mean": np.nanmean(fold_metrics[:, 8]),
            "cv_fn_mean": np.nanmean(fold_metrics[:, 9]),
        })

        if (it + 1) % PROGRESS_EVERY == 0:
            print(f"   Iter {it + 1}/{N_ITER}")

    # Guardado parcial por signo
    df_signo = pd.DataFrame(resultados_signo)
    df_signo.to_csv(CARPETA_OUT / f"{MODELO_ID}_detalle_{signo}.csv", index=False)

    resultados.extend(resultados_signo)

    cerrar_medicion(
        etapa=f"Entrenamiento y evaluación - {signo}",
        inicio_tiempo=inicio_signo,
        inicio_memoria=memoria_signo,
        modelo=MODELO_ID,
        observacion=f"{N_ITER} iteraciones, CV={K_FOLDS}, N_SAMPLE={N_SAMPLE}"
    )

cerrar_medicion(
    etapa="Entrenamiento y evaluación total",
    inicio_tiempo=inicio_etapa,
    inicio_memoria=memoria_etapa,
    modelo=MODELO_ID,
    observacion=f"{len(cols_signo)} signos, {N_ITER} iteraciones por signo, CV={K_FOLDS}, N_SAMPLE={N_SAMPLE}"
)


# =========================================================
# GUARDADO FINAL DE RESULTADOS DEL MODELO
# =========================================================
inicio_etapa, memoria_etapa = iniciar_medicion()

df_detalle = pd.DataFrame(resultados)

detalle_path = CARPETA_OUT / f"{MODELO_ID}_detalle.csv"
resumen_path = CARPETA_OUT / f"{MODELO_ID}_resumen.csv"

df_detalle.to_csv(detalle_path, index=False)

cols_metricas = [
    "pos_rate_bootstrap",
    "cv_accuracy",
    "cv_precision",
    "cv_recall",
    "cv_f1",
    "cv_roc_auc",
    "cv_pr_auc",
    "cv_tp_mean",
    "cv_fp_mean",
    "cv_tn_mean",
    "cv_fn_mean"
]

if not df_detalle.empty:
    df_resumen = (
        df_detalle
        .groupby("signo")[cols_metricas]
        .agg(["mean", "std"])
    )

    df_resumen.to_csv(resumen_path)

    print(f"{MODELO_ID.upper()} terminado")
    print("Detalle:", detalle_path)
    print("Resumen:", resumen_path)

else:
    print("No hubo resultados para guardar.")

cerrar_medicion(
    etapa="Guardado final de resultados",
    inicio_tiempo=inicio_etapa,
    inicio_memoria=memoria_etapa,
    modelo=MODELO_ID,
    observacion="Exportación de detalle y resumen"
)


# =========================================================
# GUARDADO MÉTRICAS DE SISTEMA
# =========================================================
cerrar_medicion(
    etapa="Pipeline completo",
    inicio_tiempo=inicio_total,
    inicio_memoria=memoria_total_inicio,
    modelo=MODELO_ID,
    observacion="Tiempo total desde configuración hasta guardado final"
)

df_metricas_sistema = pd.DataFrame(metricas_sistema)

# Guardado dentro de la carpeta del modelo
metricas_path_modelo = CARPETA_OUT / f"{MODELO_ID}_metricas_sistema.csv"
df_metricas_sistema.to_csv(
    metricas_path_modelo,
    index=False,
    encoding="utf-8-sig"
)

# Guardado dentro de carpeta común para comparación entre modelos
metricas_path_comun = CARPETA_METRICAS_SISTEMA / f"{MODELO_ID}_metricas_sistema.csv"
df_metricas_sistema.to_csv(
    metricas_path_comun,
    index=False,
    encoding="utf-8-sig"
)


# =========================================================
# RESUMEN DE MÉTRICAS DE SISTEMA PARA ANEXO
# =========================================================
n_signos_procesados = df_detalle["signo"].nunique() if not df_detalle.empty else 0
n_observaciones_resultado = len(df_detalle)
n_iteraciones_totales = n_signos_procesados * N_ITER

tiempo_entrenamiento_total_seg = obtener_valor_etapa(
    df_metricas_sistema,
    "Entrenamiento y evaluación total",
    "tiempo_segundos"
)

tiempo_pipeline_total_seg = obtener_valor_etapa(
    df_metricas_sistema,
    "Pipeline completo",
    "tiempo_segundos"
)

memoria_inicio_pipeline_gb = obtener_valor_etapa(
    df_metricas_sistema,
    "Pipeline completo",
    "memoria_inicio_gb"
)

memoria_fin_pipeline_gb = obtener_valor_etapa(
    df_metricas_sistema,
    "Pipeline completo",
    "memoria_fin_gb"
)

diferencia_memoria_pipeline_gb = obtener_valor_etapa(
    df_metricas_sistema,
    "Pipeline completo",
    "diferencia_memoria_gb"
)

tiempo_promedio_iteracion_seg = (
    tiempo_entrenamiento_total_seg / n_iteraciones_totales
    if n_iteraciones_totales > 0 else np.nan
)

df_resumen_sistema = pd.DataFrame([{
    "modelo": MODELO_ID,
    "modelo_nombre": MODELO_NOMBRE,
    "n_sample": N_SAMPLE,
    "n_iter_por_signo": N_ITER,
    "k_folds": K_FOLDS,
    "n_signos_procesados": n_signos_procesados,
    "n_iteraciones_totales": n_iteraciones_totales,
    "n_observaciones_resultado": n_observaciones_resultado,
    "tiempo_entrenamiento_total_seg": round(tiempo_entrenamiento_total_seg, 2),
    "tiempo_entrenamiento_total_min": round(tiempo_entrenamiento_total_seg / 60, 2),
    "tiempo_pipeline_total_seg": round(tiempo_pipeline_total_seg, 2),
    "tiempo_pipeline_total_min": round(tiempo_pipeline_total_seg / 60, 2),
    "tiempo_promedio_iteracion_seg": round(tiempo_promedio_iteracion_seg, 4),
    "memoria_inicio_pipeline_gb": memoria_inicio_pipeline_gb,
    "memoria_fin_pipeline_gb": memoria_fin_pipeline_gb,
    "diferencia_memoria_pipeline_gb": diferencia_memoria_pipeline_gb,
    "observacion": "Métricas calculadas a partir de tiempo de ejecución y memoria RSS observada por proceso."
}])

resumen_path_modelo = CARPETA_OUT / f"{MODELO_ID}_metricas_sistema_resumen.csv"
resumen_path_comun = CARPETA_METRICAS_SISTEMA / f"{MODELO_ID}_metricas_sistema_resumen.csv"

df_resumen_sistema.to_csv(
    resumen_path_modelo,
    index=False,
    encoding="utf-8-sig"
)

df_resumen_sistema.to_csv(
    resumen_path_comun,
    index=False,
    encoding="utf-8-sig"
)

print("\nMétricas de sistema guardadas en:")
print(metricas_path_modelo)
print(metricas_path_comun)

print("\nResumen de métricas de sistema guardado en:")
print(resumen_path_modelo)
print(resumen_path_comun)

display(df_metricas_sistema)
display(df_resumen_sistema)


Dataset cargado: (5808498, 488)
[Carga y preparación de datos] Tiempo: 4.94 min | Memoria inicio: 0.1883 GB | Memoria fin: 12.0214 GB

RF -> signo_zodiacal_acuario
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500
   Iter 375/500
   Iter 400/500
   Iter 425/500
   Iter 450/500
   Iter 475/500
   Iter 500/500
[Entrenamiento y evaluación - signo_zodiacal_acuario] Tiempo: 8.69 min | Memoria inicio: 12.0245 GB | Memoria fin: 10.5817 GB

RF -> signo_zodiacal_aries
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500
   Iter 375/500
   Iter 400/500
   Iter 425/500
   Iter 450/500
   Iter 475/500
   Iter 500/500
[Entrenamiento y evaluación - signo_zodiacal_ari

,modelo,etapa,tiempo_segundos,tiempo_minutos,memoria_inicio_gb,memoria_fin_gb,diferencia_memoria_gb,observacion
0,rf,Carga y preparación de datos,296.33,4.94,0.1883,12.0214,11.8331,"Dataset (5808498, 488), N_SAMPLE=100, N_ITER=5..."
1,rf,Entrenamiento y evaluación - signo_zodiacal_ac...,521.63,8.69,12.0245,10.5817,-1.4428,"500 iteraciones, CV=5, N_SAMPLE=100"
2,rf,Entrenamiento y evaluación - signo_zodiacal_aries,509.27,8.49,10.5817,0.7735,-9.8083,"500 iteraciones, CV=5, N_SAMPLE=100"
3,rf,Entrenamiento y evaluación - signo_zodiacal_ca...,502.67,8.38,0.7735,1.0327,0.2592,"500 iteraciones, CV=5, N_SAMPLE=100"
4,rf,Entrenamiento y evaluación - signo_zodiacal_ca...,502.44,8.37,1.0327,1.2833,0.2506,"500 iteraciones, CV=5, N_SAMPLE=100"
5,rf,Entrenamiento y evaluación - signo_zodiacal_es...,501.59,8.36,1.2833,1.5254,0.2421,"500 iteraciones, CV=5, N_SAMPLE=100"
6,rf,Entrenamiento y evaluación - signo_zodiacal_ge...,502.77,8.38,1.5254,1.7616,0.2362,"500 iteraciones, CV=5, N_SAMPLE=100"
7,rf,Entrenamiento y evaluación - signo_zodiacal_leo,502.24,8.37,1.7616,1.9896,0.2280,"500 iteraciones, CV=5, N_SAMPLE=100"
8,rf,Entrenamiento y evaluación - signo_zodiacal_libra,501.09,8.35,1.9896,2.2112,0.2216,"500 iteraciones, CV=5, N_SAMPLE=100"
9,rf,Entrenamiento y evaluación - signo_zodiacal_pi...,501.80,8.36,2.2112,2.4249,0.2137,"500 iteraciones, CV=5, N_SAMPLE=100"


,modelo,modelo_nombre,n_sample,n_iter_por_signo,k_folds,n_signos_procesados,n_iteraciones_totales,n_observaciones_resultado,tiempo_entrenamiento_total_seg,tiempo_entrenamiento_total_min,tiempo_pipeline_total_seg,tiempo_pipeline_total_min,tiempo_promedio_iteracion_seg,memoria_inicio_pipeline_gb,memoria_fin_pipeline_gb,diferencia_memoria_pipeline_gb,observacion
0,rf,Random Forest,100,500,5,12,6000,6000,6050.09,100.83,6346.62,105.78,1.0083,0.1883,2.9954,2.8071,Métricas calculadas a partir de tiempo de ejec...


# Métricas KNN

In [1]:

# =========================================================
# K-NEAREST NEIGHBORS - BOOTSTRAPPING + MÉTRICAS DE SISTEMA
# =========================================================

import pandas as pd
import numpy as np
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone

# =========================================================
# MÉTRICAS DE SISTEMA
# =========================================================
from time import perf_counter
import os
import psutil

process = psutil.Process(os.getpid())
metricas_sistema = []


def memoria_gb():
    """
    Retorna la memoria RSS utilizada por el proceso actual, expresada en GB.
    """
    return process.memory_info().rss / 1024 / 1024 / 1024


def iniciar_medicion():
    """
    Inicia una medición de tiempo y memoria.
    """
    return perf_counter(), memoria_gb()


def cerrar_medicion(etapa, inicio_tiempo, inicio_memoria, modelo="knn", observacion=""):
    """
    Cierra una medición de tiempo y memoria para una etapa específica.
    Guarda la información en la lista metricas_sistema.
    """
    fin_tiempo = perf_counter()
    fin_memoria = memoria_gb()

    tiempo_segundos = fin_tiempo - inicio_tiempo

    fila = {
        "modelo": modelo,
        "etapa": etapa,
        "tiempo_segundos": round(tiempo_segundos, 2),
        "tiempo_minutos": round(tiempo_segundos / 60, 2),
        "memoria_inicio_gb": round(inicio_memoria, 4),
        "memoria_fin_gb": round(fin_memoria, 4),
        "diferencia_memoria_gb": round(fin_memoria - inicio_memoria, 4),
        "observacion": observacion
    }

    metricas_sistema.append(fila)

    print(
        f"[{etapa}] Tiempo: {fila['tiempo_minutos']} min | "
        f"Memoria inicio: {fila['memoria_inicio_gb']} GB | "
        f"Memoria fin: {fila['memoria_fin_gb']} GB"
    )


# =========================================================
# CONFIG
# =========================================================
RUTA_DATASET = "../1_data_processed/v3_dataset_post_chi.parquet"
RUTA_SPLIT = Path("../1_data_processed/split_indices")

N_SAMPLE = 100
N_ITER = 500
K_FOLDS = 5
BASE_SEED = 42
PROGRESS_EVERY = 25

MODELO_ID = "knn"
MODELO_NOMBRE = "K-Nearest Neighbors"

CARPETA_OUT = Path("../4_results/modelo_knn")
CARPETA_OUT.mkdir(parents=True, exist_ok=True)

CARPETA_METRICAS_SISTEMA = Path("../4_results/metricas_sistema")
CARPETA_METRICAS_SISTEMA.mkdir(parents=True, exist_ok=True)

MODELO = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(
        n_neighbors=50,
        weights="uniform",
        metric="euclidean",
        n_jobs=1
    ))
])

# Medición total del pipeline
inicio_total, memoria_total_inicio = iniciar_medicion()


# =========================================================
# HELPERS
# =========================================================
def evaluar_modelo(modelo, X_eval, y_eval):
    """
    Evalúa un modelo binario y retorna métricas de clasificación.
    """
    y_pred = modelo.predict(X_eval)

    tn, fp, fn, tp = confusion_matrix(y_eval, y_pred, labels=[0, 1]).ravel()

    acc = accuracy_score(y_eval, y_pred)
    prec = precision_score(y_eval, y_pred, zero_division=0)
    rec = recall_score(y_eval, y_pred, zero_division=0)
    f1 = f1_score(y_eval, y_pred, zero_division=0)

    roc = np.nan
    pr = np.nan

    try:
        if hasattr(modelo, "predict_proba"):
            y_proba = modelo.predict_proba(X_eval)[:, 1]
            roc = roc_auc_score(y_eval, y_proba)
            pr = average_precision_score(y_eval, y_proba)
    except Exception:
        pass

    return acc, prec, rec, f1, roc, pr, tp, fp, tn, fn


def obtener_valor_etapa(df_metricas, etapa, columna):
    """
    Obtiene el valor de una columna para una etapa específica.
    Si la etapa no existe, retorna NaN.
    """
    fila = df_metricas[df_metricas["etapa"] == etapa]

    if fila.empty:
        return np.nan

    return fila.iloc[0][columna]


# =========================================================
# CARGA Y PREPARACIÓN
# =========================================================
inicio_etapa, memoria_etapa = iniciar_medicion()

for ext_name in ["pandas.period", "pandas.interval"]:
    try:
        pa.unregister_extension_type(ext_name)
    except Exception:
        pass

cols = pq.read_schema(RUTA_DATASET).names
cols_signo = [c for c in cols if c.startswith("signo_zodiacal_")]

if not cols_signo:
    raise ValueError("No se encontraron columnas signo_zodiacal_*.")

df = pd.read_parquet(RUTA_DATASET, engine="pyarrow")
print("Dataset cargado:", df.shape)

# Compactar tipos
for c in df.columns:
    if c.startswith("signo_zodiacal_"):
        df[c] = df[c].astype("uint8")
    elif c == "ESTANCIA_DIAS":
        df[c] = df[c].astype("int32")
    elif pd.api.types.is_numeric_dtype(df[c]):
        df[c] = df[c].astype("uint8")

idx_train = np.load(RUTA_SPLIT / "idx_train.npy")

# Pasar a NumPy para acelerar el loop
X_all = df.drop(columns=cols_signo).to_numpy(copy=False)
df_train = df.iloc[idx_train].reset_index(drop=True)
X_train_all = X_all[idx_train]

skf = StratifiedKFold(
    n_splits=K_FOLDS,
    shuffle=True,
    random_state=BASE_SEED
)

cerrar_medicion(
    etapa="Carga y preparación de datos",
    inicio_tiempo=inicio_etapa,
    inicio_memoria=memoria_etapa,
    modelo=MODELO_ID,
    observacion=f"Dataset {df.shape}, N_SAMPLE={N_SAMPLE}, N_ITER={N_ITER}, K_FOLDS={K_FOLDS}"
)


# =========================================================
# ENTRENAMIENTO Y EVALUACIÓN
# =========================================================
inicio_etapa, memoria_etapa = iniciar_medicion()

resultados = []

for signo in cols_signo:
    inicio_signo, memoria_signo = iniciar_medicion()

    print(f"\n{MODELO_ID.upper()} -> {signo}")

    y_train_all = df_train[signo].to_numpy(dtype=np.uint8)

    pos_idx = np.where(y_train_all == 1)[0]
    neg_idx = np.where(y_train_all == 0)[0]

    if len(pos_idx) == 0 or len(neg_idx) == 0:
        print(f"Saltando {signo}")

        cerrar_medicion(
            etapa=f"Entrenamiento y evaluación - {signo}",
            inicio_tiempo=inicio_signo,
            inicio_memoria=memoria_signo,
            modelo=MODELO_ID,
            observacion="Signo saltado por ausencia de clase positiva o negativa"
        )

        continue

    n_pos = N_SAMPLE // 2
    n_neg = N_SAMPLE - n_pos

    resultados_signo = []

    for it in range(N_ITER):
        rng = np.random.default_rng(BASE_SEED + it + abs(hash(signo)) % 10000)

        # Bootstrap balanceado
        sample_pos = rng.choice(pos_idx, size=n_pos, replace=True)
        sample_neg = rng.choice(neg_idx, size=n_neg, replace=True)

        sample_idx = np.concatenate([sample_pos, sample_neg])
        rng.shuffle(sample_idx)

        X_boot = X_train_all[sample_idx]
        y_boot = y_train_all[sample_idx]

        fold_metrics = []

        # Validación cruzada k-fold
        for tr_idx, val_idx in skf.split(X_boot, y_boot):
            X_tr, X_val = X_boot[tr_idx], X_boot[val_idx]
            y_tr, y_val = y_boot[tr_idx], y_boot[val_idx]

            m_cv = clone(MODELO)
            m_cv.fit(X_tr, y_tr)

            fold_metrics.append(evaluar_modelo(m_cv, X_val, y_val))

        fold_metrics = np.array(fold_metrics, dtype=float)

        resultados_signo.append({
            "modelo": MODELO_ID,
            "signo": signo,
            "iter": it + 1,
            "pos_rate_bootstrap": float(y_boot.mean()),
            "cv_accuracy": np.nanmean(fold_metrics[:, 0]),
            "cv_precision": np.nanmean(fold_metrics[:, 1]),
            "cv_recall": np.nanmean(fold_metrics[:, 2]),
            "cv_f1": np.nanmean(fold_metrics[:, 3]),
            "cv_roc_auc": np.nanmean(fold_metrics[:, 4]),
            "cv_pr_auc": np.nanmean(fold_metrics[:, 5]),
            "cv_tp_mean": np.nanmean(fold_metrics[:, 6]),
            "cv_fp_mean": np.nanmean(fold_metrics[:, 7]),
            "cv_tn_mean": np.nanmean(fold_metrics[:, 8]),
            "cv_fn_mean": np.nanmean(fold_metrics[:, 9])
        })

        if (it + 1) % PROGRESS_EVERY == 0:
            print(f"   Iter {it + 1}/{N_ITER}")

    # Guardado parcial por signo
    df_signo = pd.DataFrame(resultados_signo)
    df_signo.to_csv(CARPETA_OUT / f"{MODELO_ID}_detalle_{signo}.csv", index=False)

    resultados.extend(resultados_signo)

    cerrar_medicion(
        etapa=f"Entrenamiento y evaluación - {signo}",
        inicio_tiempo=inicio_signo,
        inicio_memoria=memoria_signo,
        modelo=MODELO_ID,
        observacion=f"{N_ITER} iteraciones, CV={K_FOLDS}, N_SAMPLE={N_SAMPLE}"
    )

cerrar_medicion(
    etapa="Entrenamiento y evaluación total",
    inicio_tiempo=inicio_etapa,
    inicio_memoria=memoria_etapa,
    modelo=MODELO_ID,
    observacion=f"{len(cols_signo)} signos, {N_ITER} iteraciones por signo, CV={K_FOLDS}, N_SAMPLE={N_SAMPLE}"
)


# =========================================================
# GUARDADO FINAL DE RESULTADOS DEL MODELO
# =========================================================
inicio_etapa, memoria_etapa = iniciar_medicion()

df_detalle = pd.DataFrame(resultados)

detalle_path = CARPETA_OUT / f"{MODELO_ID}_detalle.csv"
resumen_path = CARPETA_OUT / f"{MODELO_ID}_resumen.csv"

df_detalle.to_csv(detalle_path, index=False)

cols_metricas = [
    "pos_rate_bootstrap",
    "cv_accuracy",
    "cv_precision",
    "cv_recall",
    "cv_f1",
    "cv_roc_auc",
    "cv_pr_auc",
    "cv_tp_mean",
    "cv_fp_mean",
    "cv_tn_mean",
    "cv_fn_mean"
]

if not df_detalle.empty:
    df_resumen = (
        df_detalle
        .groupby("signo")[cols_metricas]
        .agg(["mean", "std"])
    )

    df_resumen.to_csv(resumen_path)

    print(f"{MODELO_ID.upper()} terminado")
    print("Detalle:", detalle_path)
    print("Resumen:", resumen_path)

else:
    print("No hubo resultados para guardar.")

cerrar_medicion(
    etapa="Guardado final de resultados",
    inicio_tiempo=inicio_etapa,
    inicio_memoria=memoria_etapa,
    modelo=MODELO_ID,
    observacion="Exportación de detalle y resumen"
)


# =========================================================
# GUARDADO MÉTRICAS DE SISTEMA
# =========================================================
cerrar_medicion(
    etapa="Pipeline completo",
    inicio_tiempo=inicio_total,
    inicio_memoria=memoria_total_inicio,
    modelo=MODELO_ID,
    observacion="Tiempo total desde configuración hasta guardado final"
)

df_metricas_sistema = pd.DataFrame(metricas_sistema)

# Guardado dentro de la carpeta del modelo
metricas_path_modelo = CARPETA_OUT / f"{MODELO_ID}_metricas_sistema.csv"
df_metricas_sistema.to_csv(
    metricas_path_modelo,
    index=False,
    encoding="utf-8-sig"
)

# Guardado dentro de carpeta común para comparación entre modelos
metricas_path_comun = CARPETA_METRICAS_SISTEMA / f"{MODELO_ID}_metricas_sistema.csv"
df_metricas_sistema.to_csv(
    metricas_path_comun,
    index=False,
    encoding="utf-8-sig"
)


# =========================================================
# RESUMEN DE MÉTRICAS DE SISTEMA PARA ANEXO
# =========================================================
n_signos_procesados = df_detalle["signo"].nunique() if not df_detalle.empty else 0
n_observaciones_resultado = len(df_detalle)
n_iteraciones_totales = n_signos_procesados * N_ITER

tiempo_entrenamiento_total_seg = obtener_valor_etapa(
    df_metricas_sistema,
    "Entrenamiento y evaluación total",
    "tiempo_segundos"
)

tiempo_pipeline_total_seg = obtener_valor_etapa(
    df_metricas_sistema,
    "Pipeline completo",
    "tiempo_segundos"
)

memoria_inicio_pipeline_gb = obtener_valor_etapa(
    df_metricas_sistema,
    "Pipeline completo",
    "memoria_inicio_gb"
)

memoria_fin_pipeline_gb = obtener_valor_etapa(
    df_metricas_sistema,
    "Pipeline completo",
    "memoria_fin_gb"
)

diferencia_memoria_pipeline_gb = obtener_valor_etapa(
    df_metricas_sistema,
    "Pipeline completo",
    "diferencia_memoria_gb"
)

tiempo_promedio_iteracion_seg = (
    tiempo_entrenamiento_total_seg / n_iteraciones_totales
    if n_iteraciones_totales > 0 else np.nan
)

df_resumen_sistema = pd.DataFrame([{
    "modelo": MODELO_ID,
    "modelo_nombre": MODELO_NOMBRE,
    "n_sample": N_SAMPLE,
    "n_iter_por_signo": N_ITER,
    "k_folds": K_FOLDS,
    "n_signos_procesados": n_signos_procesados,
    "n_iteraciones_totales": n_iteraciones_totales,
    "n_observaciones_resultado": n_observaciones_resultado,
    "tiempo_entrenamiento_total_seg": round(tiempo_entrenamiento_total_seg, 2),
    "tiempo_entrenamiento_total_min": round(tiempo_entrenamiento_total_seg / 60, 2),
    "tiempo_pipeline_total_seg": round(tiempo_pipeline_total_seg, 2),
    "tiempo_pipeline_total_min": round(tiempo_pipeline_total_seg / 60, 2),
    "tiempo_promedio_iteracion_seg": round(tiempo_promedio_iteracion_seg, 4),
    "memoria_inicio_pipeline_gb": memoria_inicio_pipeline_gb,
    "memoria_fin_pipeline_gb": memoria_fin_pipeline_gb,
    "diferencia_memoria_pipeline_gb": diferencia_memoria_pipeline_gb,
    "observacion": "Métricas calculadas a partir de tiempo de ejecución y memoria RSS observada por proceso."
}])

resumen_path_modelo = CARPETA_OUT / f"{MODELO_ID}_metricas_sistema_resumen.csv"
resumen_path_comun = CARPETA_METRICAS_SISTEMA / f"{MODELO_ID}_metricas_sistema_resumen.csv"

df_resumen_sistema.to_csv(
    resumen_path_modelo,
    index=False,
    encoding="utf-8-sig"
)

df_resumen_sistema.to_csv(
    resumen_path_comun,
    index=False,
    encoding="utf-8-sig"
)

print("\nMétricas de sistema guardadas en:")
print(metricas_path_modelo)
print(metricas_path_comun)

print("\nResumen de métricas de sistema guardado en:")
print(resumen_path_modelo)
print(resumen_path_comun)

display(df_metricas_sistema)
display(df_resumen_sistema)


Dataset cargado: (5808498, 488)
[Carga y preparación de datos] Tiempo: 4.06 min | Memoria inicio: 0.1848 GB | Memoria fin: 12.1155 GB

KNN -> signo_zodiacal_acuario
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500
   Iter 375/500
   Iter 400/500
   Iter 425/500
   Iter 450/500
   Iter 475/500
   Iter 500/500
[Entrenamiento y evaluación - signo_zodiacal_acuario] Tiempo: 0.84 min | Memoria inicio: 12.1185 GB | Memoria fin: 11.7893 GB

KNN -> signo_zodiacal_aries
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500
   Iter 375/500
   Iter 400/500
   Iter 425/500
   Iter 450/500
   Iter 475/500
   Iter 500/500
[Entrenamiento y evaluación - signo_zodiacal_a

,modelo,etapa,tiempo_segundos,tiempo_minutos,memoria_inicio_gb,memoria_fin_gb,diferencia_memoria_gb,observacion
0,knn,Carga y preparación de datos,243.44,4.06,0.1848,12.1155,11.9307,"Dataset (5808498, 488), N_SAMPLE=100, N_ITER=5..."
1,knn,Entrenamiento y evaluación - signo_zodiacal_ac...,50.22,0.84,12.1185,11.7893,-0.3291,"500 iteraciones, CV=5, N_SAMPLE=100"
2,knn,Entrenamiento y evaluación - signo_zodiacal_aries,48.55,0.81,11.7893,11.6647,-0.1247,"500 iteraciones, CV=5, N_SAMPLE=100"
3,knn,Entrenamiento y evaluación - signo_zodiacal_ca...,48.87,0.81,11.6647,11.4852,-0.1794,"500 iteraciones, CV=5, N_SAMPLE=100"
4,knn,Entrenamiento y evaluación - signo_zodiacal_ca...,48.47,0.81,11.4853,11.3606,-0.1247,"500 iteraciones, CV=5, N_SAMPLE=100"
5,knn,Entrenamiento y evaluación - signo_zodiacal_es...,48.01,0.80,11.3606,11.1374,-0.2231,"500 iteraciones, CV=5, N_SAMPLE=100"
6,knn,Entrenamiento y evaluación - signo_zodiacal_ge...,52.34,0.87,11.1374,10.6500,-0.4875,"500 iteraciones, CV=5, N_SAMPLE=100"
7,knn,Entrenamiento y evaluación - signo_zodiacal_leo,47.34,0.79,10.6500,10.6083,-0.0417,"500 iteraciones, CV=5, N_SAMPLE=100"
8,knn,Entrenamiento y evaluación - signo_zodiacal_libra,45.78,0.76,10.6083,10.5599,-0.0484,"500 iteraciones, CV=5, N_SAMPLE=100"
9,knn,Entrenamiento y evaluación - signo_zodiacal_pi...,45.29,0.75,10.5599,10.5042,-0.0557,"500 iteraciones, CV=5, N_SAMPLE=100"


,modelo,modelo_nombre,n_sample,n_iter_por_signo,k_folds,n_signos_procesados,n_iteraciones_totales,n_observaciones_resultado,tiempo_entrenamiento_total_seg,tiempo_entrenamiento_total_min,tiempo_pipeline_total_seg,tiempo_pipeline_total_min,tiempo_promedio_iteracion_seg,memoria_inicio_pipeline_gb,memoria_fin_pipeline_gb,diferencia_memoria_pipeline_gb,observacion
0,knn,K-Nearest Neighbors,100,500,5,12,6000,6000,564.38,9.41,808.02,13.47,0.0941,0.1848,10.5621,10.3772,Métricas calculadas a partir de tiempo de ejec...


# Conclusión

In [1]:

# =========================================================
# CONSOLIDACIÓN DE MÉTRICAS DE SISTEMA POR MODELO
# =========================================================

import pandas as pd
import numpy as np
from pathlib import Path


# =========================================================
# CONFIGURACIÓN
# =========================================================
# Este notebook se ejecuta desde:
# 3_notebooks/

RUTA_BASE = Path("..")

CARPETA_METRICAS = RUTA_BASE / "4_results" / "metricas_sistema"
CARPETA_SALIDA = CARPETA_METRICAS
CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)

ARCHIVOS_MODELOS = {
    "dt": {
        "nombre": "Árbol de Decisión",
        "ruta": CARPETA_METRICAS / "dt_metricas_sistema.csv"
    },
    "lr": {
        "nombre": "Regresión Logística",
        "ruta": CARPETA_METRICAS / "lr_metricas_sistema.csv"
    },
    "rf": {
        "nombre": "Random Forest",
        "ruta": CARPETA_METRICAS / "rf_metricas_sistema.csv"
    },
    "knn": {
        "nombre": "K-Nearest Neighbors",
        "ruta": CARPETA_METRICAS / "knn_metricas_sistema.csv"
    }
}

RUTA_CONSOLIDADO_DETALLE = CARPETA_SALIDA / "metricas_sistema_consolidado_detalle.csv"
RUTA_CONSOLIDADO_RESUMEN = CARPETA_SALIDA / "metricas_sistema_consolidado_resumen.csv"
RUTA_CONSOLIDADO_ETAPAS = CARPETA_SALIDA / "metricas_sistema_consolidado_etapas.csv"


# =========================================================
# CARGA DE ARCHIVOS
# =========================================================
dfs = []

for modelo_id, info in ARCHIVOS_MODELOS.items():
    ruta = info["ruta"]
    nombre_modelo = info["nombre"]

    if not ruta.exists():
        print(f"Archivo no encontrado para {nombre_modelo}: {ruta}")
        continue

    df = pd.read_csv(ruta)

    df["modelo"] = modelo_id
    df["modelo_nombre"] = nombre_modelo

    dfs.append(df)

if not dfs:
    raise FileNotFoundError("No se encontró ningún archivo de métricas de sistema.")

df_metricas = pd.concat(dfs, ignore_index=True)


# =========================================================
# ORDEN Y LIMPIEZA
# =========================================================
columnas_orden = [
    "modelo",
    "modelo_nombre",
    "etapa",
    "tiempo_segundos",
    "tiempo_minutos",
    "memoria_inicio_gb",
    "memoria_fin_gb",
    "diferencia_memoria_gb",
    "observacion"
]

columnas_existentes = [c for c in columnas_orden if c in df_metricas.columns]
df_metricas = df_metricas[columnas_existentes]

# Orden lógico por modelo y etapa
orden_modelos = {
    "dt": 1,
    "lr": 2,
    "rf": 3,
    "knn": 4
}

df_metricas["orden_modelo"] = df_metricas["modelo"].map(orden_modelos)

df_metricas = (
    df_metricas
    .sort_values(["orden_modelo", "etapa"])
    .drop(columns=["orden_modelo"])
    .reset_index(drop=True)
)


# =========================================================
# CONSOLIDADO DETALLADO
# =========================================================
df_metricas.to_csv(
    RUTA_CONSOLIDADO_DETALLE,
    index=False,
    encoding="utf-8-sig"
)

print("Consolidado detallado guardado en:")
print(RUTA_CONSOLIDADO_DETALLE)


# =========================================================
# RESUMEN PRINCIPAL POR MODELO
# =========================================================
def obtener_valor(df, etapa, columna):
    fila = df[df["etapa"] == etapa]

    if fila.empty or columna not in df.columns:
        return np.nan

    return fila.iloc[0][columna]


resumen_modelos = []

for modelo_id, df_modelo in df_metricas.groupby("modelo"):
    nombre_modelo = df_modelo["modelo_nombre"].iloc[0]

    tiempo_carga_seg = obtener_valor(
        df_modelo,
        "Carga y preparación de datos",
        "tiempo_segundos"
    )

    tiempo_entrenamiento_seg = obtener_valor(
        df_modelo,
        "Entrenamiento y evaluación total",
        "tiempo_segundos"
    )

    tiempo_guardado_seg = obtener_valor(
        df_modelo,
        "Guardado final de resultados",
        "tiempo_segundos"
    )

    tiempo_pipeline_seg = obtener_valor(
        df_modelo,
        "Pipeline completo",
        "tiempo_segundos"
    )

    memoria_inicio_pipeline_gb = obtener_valor(
        df_modelo,
        "Pipeline completo",
        "memoria_inicio_gb"
    )

    memoria_fin_pipeline_gb = obtener_valor(
        df_modelo,
        "Pipeline completo",
        "memoria_fin_gb"
    )

    diferencia_memoria_pipeline_gb = obtener_valor(
        df_modelo,
        "Pipeline completo",
        "diferencia_memoria_gb"
    )

    # Cantidad de etapas por signo medidas
    etapas_signo = df_modelo[
        df_modelo["etapa"].astype(str).str.startswith("Entrenamiento y evaluación - signo_zodiacal_")
    ]

    n_signos_medidos = len(etapas_signo)

    tiempo_promedio_por_signo_seg = (
        etapas_signo["tiempo_segundos"].mean()
        if not etapas_signo.empty else np.nan
    )

    tiempo_std_por_signo_seg = (
        etapas_signo["tiempo_segundos"].std()
        if not etapas_signo.empty else np.nan
    )

    tiempo_min_por_signo_seg = (
        etapas_signo["tiempo_segundos"].min()
        if not etapas_signo.empty else np.nan
    )

    tiempo_max_por_signo_seg = (
        etapas_signo["tiempo_segundos"].max()
        if not etapas_signo.empty else np.nan
    )

    resumen_modelos.append({
        "modelo": modelo_id,
        "modelo_nombre": nombre_modelo,
        "n_signos_medidos": n_signos_medidos,

        "tiempo_carga_seg": tiempo_carga_seg,
        "tiempo_carga_min": tiempo_carga_seg / 60 if pd.notna(tiempo_carga_seg) else np.nan,

        "tiempo_entrenamiento_total_seg": tiempo_entrenamiento_seg,
        "tiempo_entrenamiento_total_min": tiempo_entrenamiento_seg / 60 if pd.notna(tiempo_entrenamiento_seg) else np.nan,

        "tiempo_guardado_seg": tiempo_guardado_seg,
        "tiempo_guardado_min": tiempo_guardado_seg / 60 if pd.notna(tiempo_guardado_seg) else np.nan,

        "tiempo_pipeline_total_seg": tiempo_pipeline_seg,
        "tiempo_pipeline_total_min": tiempo_pipeline_seg / 60 if pd.notna(tiempo_pipeline_seg) else np.nan,

        "tiempo_promedio_por_signo_seg": tiempo_promedio_por_signo_seg,
        "tiempo_std_por_signo_seg": tiempo_std_por_signo_seg,
        "tiempo_min_por_signo_seg": tiempo_min_por_signo_seg,
        "tiempo_max_por_signo_seg": tiempo_max_por_signo_seg,

        "memoria_inicio_pipeline_gb": memoria_inicio_pipeline_gb,
        "memoria_fin_pipeline_gb": memoria_fin_pipeline_gb,
        "diferencia_memoria_pipeline_gb": diferencia_memoria_pipeline_gb
    })

df_resumen = pd.DataFrame(resumen_modelos)

df_resumen["orden_modelo"] = df_resumen["modelo"].map(orden_modelos)

df_resumen = (
    df_resumen
    .sort_values("orden_modelo")
    .drop(columns=["orden_modelo"])
    .reset_index(drop=True)
)

# Redondear columnas numéricas
for col in df_resumen.select_dtypes(include=["float", "int"]).columns:
    df_resumen[col] = df_resumen[col].round(4)

df_resumen.to_csv(
    RUTA_CONSOLIDADO_RESUMEN,
    index=False,
    encoding="utf-8-sig"
)

print("\nResumen por modelo guardado en:")
print(RUTA_CONSOLIDADO_RESUMEN)


# =========================================================
# TABLA POR ETAPA PARA ANEXO
# =========================================================
etapas_interes = [
    "Carga y preparación de datos",
    "Entrenamiento y evaluación total",
    "Guardado final de resultados",
    "Pipeline completo"
]

df_etapas = df_metricas[df_metricas["etapa"].isin(etapas_interes)].copy()

df_etapas = df_etapas[[
    "modelo",
    "modelo_nombre",
    "etapa",
    "tiempo_segundos",
    "tiempo_minutos",
    "memoria_inicio_gb",
    "memoria_fin_gb",
    "diferencia_memoria_gb"
]]

df_etapas["orden_modelo"] = df_etapas["modelo"].map(orden_modelos)
df_etapas["orden_etapa"] = df_etapas["etapa"].map({
    "Carga y preparación de datos": 1,
    "Entrenamiento y evaluación total": 2,
    "Guardado final de resultados": 3,
    "Pipeline completo": 4
})

df_etapas = (
    df_etapas
    .sort_values(["orden_modelo", "orden_etapa"])
    .drop(columns=["orden_modelo", "orden_etapa"])
    .reset_index(drop=True)
)

for col in df_etapas.select_dtypes(include=["float", "int"]).columns:
    df_etapas[col] = df_etapas[col].round(4)

df_etapas.to_csv(
    RUTA_CONSOLIDADO_ETAPAS,
    index=False,
    encoding="utf-8-sig"
)

print("\nTabla por etapas guardada en:")
print(RUTA_CONSOLIDADO_ETAPAS)


# =========================================================
# VISTA FINAL
# =========================================================
print("\nResumen principal:")
display(df_resumen)

print("\nDetalle por etapas principales:")
display(df_etapas)


Consolidado detallado guardado en:
..\4_results\metricas_sistema\metricas_sistema_consolidado_detalle.csv

Resumen por modelo guardado en:
..\4_results\metricas_sistema\metricas_sistema_consolidado_resumen.csv

Tabla por etapas guardada en:
..\4_results\metricas_sistema\metricas_sistema_consolidado_etapas.csv

Resumen principal:


,modelo,modelo_nombre,n_signos_medidos,tiempo_carga_seg,tiempo_carga_min,tiempo_entrenamiento_total_seg,tiempo_entrenamiento_total_min,tiempo_guardado_seg,tiempo_guardado_min,tiempo_pipeline_total_seg,tiempo_pipeline_total_min,tiempo_promedio_por_signo_seg,tiempo_std_por_signo_seg,tiempo_min_por_signo_seg,tiempo_max_por_signo_seg,memoria_inicio_pipeline_gb,memoria_fin_pipeline_gb,diferencia_memoria_pipeline_gb
0,dt,Árbol de Decisión,12,327.94,5.4657,401.68,6.6947,0.18,0.0030,729.83,12.1638,33.4708,4.4323,27.98,38.71,0.1882,6.6242,6.4360
1,lr,Regresión Logística,12,235.58,3.9263,423.92,7.0653,0.16,0.0027,659.69,10.9948,35.3225,2.3181,32.87,40.23,0.1815,5.4027,5.2212
2,rf,Random Forest,12,296.33,4.9388,6050.09,100.8348,0.11,0.0018,6346.62,105.7770,504.1658,5.9492,500.19,521.63,0.1883,2.9954,2.8071
3,knn,K-Nearest Neighbors,12,243.44,4.0573,564.38,9.4063,0.16,0.0027,808.02,13.4670,47.0267,2.9796,42.72,52.34,0.1848,10.5621,10.3772



Detalle por etapas principales:


,modelo,modelo_nombre,etapa,tiempo_segundos,tiempo_minutos,memoria_inicio_gb,memoria_fin_gb,diferencia_memoria_gb
0,dt,Árbol de Decisión,Carga y preparación de datos,327.94,5.47,0.1882,11.4079,11.2197
1,dt,Árbol de Decisión,Entrenamiento y evaluación total,401.68,6.69,11.4090,6.6158,-4.7932
2,dt,Árbol de Decisión,Guardado final de resultados,0.18,0.00,6.6159,6.6242,0.0083
3,dt,Árbol de Decisión,Pipeline completo,729.83,12.16,0.1882,6.6242,6.4360
4,lr,Regresión Logística,Carga y preparación de datos,235.58,3.93,0.1815,12.2712,12.0897
5,lr,Regresión Logística,Entrenamiento y evaluación total,423.92,7.07,12.2723,5.3941,-6.8782
6,lr,Regresión Logística,Guardado final de resultados,0.16,0.00,5.3941,5.4026,0.0085
7,lr,Regresión Logística,Pipeline completo,659.69,10.99,0.1815,5.4027,5.2212
8,rf,Random Forest,Carga y preparación de datos,296.33,4.94,0.1883,12.0214,11.8331
9,rf,Random Forest,Entrenamiento y evaluación total,6050.09,100.83,12.0229,2.9906,-9.0323
